In [2]:
# %pip install -q numpy pandas

In [42]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [ ]:
TRAIN_PATH = r"C:\Users\AYO_AYO\Desktop\Machine-Learning-Bootcamp\Data_Preprocessing\titanic_train_imputed.csv"
TEST_PATH = r"C:\Users\AYO_AYO\Desktop\Machine-Learning-Bootcamp\Data_Preprocessing\titanic_test_imputed.csv"
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

**Separate the features and target**

In [5]:
target = "survived"

X_train = train_df.drop(columns=target)
y_train = train_df[target]

X_test = test_df.drop(columns=target)
y_test = test_df[target]

In [6]:
print("X train shape", X_train.shape)
print("X test shape", X_test.shape)

X train shape (712, 9)
X test shape (179, 9)


In [7]:
X_test.head()

,pclass,sex,age,sibsp,parch,fare,adult_male,embark_town,alone
0,3,male,24.000000,2,0,24.1500,True,Southampton,False
1,3,male,44.000000,0,1,16.1000,True,Southampton,False
2,3,male,22.000000,0,0,7.2250,True,Cherbourg,True
3,3,male,41.000000,2,0,14.1083,True,Southampton,False
4,3,female,29.807687,1,0,15.5000,False,Queenstown,False


In [8]:
y_test.head()

0    0
1    0
2    1
3    0
4    1
Name: survived, dtype: int64

In [9]:
numerical_cols = X_train.select_dtypes([np.number]).columns.to_list()
categorical_cols = [x for x in X_train.columns if x not in numerical_cols]

In [10]:
print(numerical_cols)
print(categorical_cols)

['pclass', 'age', 'sibsp', 'parch', 'fare']
['sex', 'adult_male', 'embark_town', 'alone']


In [11]:
for col in categorical_cols:
    print(col, X_train[col].unique())
    print("-"*10)

sex ['male' 'female']
----------
adult_male [ True False]
----------
embark_town ['Southampton' 'Cherbourg' 'Queenstown']
----------
alone [ True False]
----------


**Encoding Separation**
- Label Encoding: Ordinal/Categorical features: When the feature follows **an order** i.e Low, Medium, High
- One Hot Encoding: Use when Categories have **no natural order**. e.g: Country, Gender, City, Product Category


| Column       | Type     | Explanation |
|--------------|----------|-------------|
| **pclass**       | Ordinal | Travel classes have a natural order (1 > 2 > 3) |
| **sex**          | Nominal (binary) | No ranking between male and female |
| **embarked**     | Nominal | Ports do not have any order |
| **who**          | Nominal *(optional ordinal)* | Usually treated as nominal unless modeling age groups |
| **adult_male**   | Nominal (binary) | Binary indicator, no inherent order |
| **deck**         | Nominal | Cabin letters do not represent a meaningful order |
| **alone**        | Nominal (binary) | Binary indicator, no hierarchy |

### **Encoding**
- This is a technique that involves converting textual/categorical datapoints into numerical value which is easily understable by the ML model

In [12]:
binary_and_ordinal_col = ["sex", "adult_male", "alone"] #label/ordinal enconding
multi_class_nominal_col = ["embark_town"] # one hot encoding

In [13]:
X_train[binary_and_ordinal_col]

,sex,adult_male,alone
0,male,True,True
1,male,True,True
2,male,True,True
3,female,False,False
4,female,False,False
...,...,...,...
707,female,False,True
708,female,False,True
709,female,False,False
710,male,True,True


### **Label Encoder**

In [14]:
# from sklearn.preprocessing import LabelEncoder

# # Rule of thumb
# # Training data: fit() then transform() (or fit_transform() if you only need the transformed training data).
# # Validation/Test/New data: only transform(). Never call fit() on these datasets, because the model should use the mapping learned from the training data.

# basket = {}

# for cols in binary_and_ordinal_col:
#     label_encoder = LabelEncoder()
#     label_encoder.fit(X_train[cols])
#     basket[cols] = label_encoder

#     X_train[cols] = label_encoder.transform(X_train[cols])
#     X_test[cols] = label_encoder.transform(X_test[cols])

# print(X_train.head(3))
# print(basket)

### **Ordinal Encoder**

In [15]:
# from sklearn.preprocessing import OrdinalEncoder

# for cols in binary_and_ordinal_col:
#     ord_encoder = OrdinalEncoder(
#     # categories= binary_and_ordinal_col, # this is not needed since the loop is for binary columns
#     handle_unknown= "use_encoded_value",
#     unknown_value=-1
#     )
    
#     ord_encoder.fit(X_train[[cols]])
    
#     X_train[[cols]] = ord_encoder.transform(X_train[[cols]])
#     X_test[[cols]] = ord_encoder.transform(X_test[[cols]])
    
# X_train.head()

### **One Hot Encoder**

In [16]:
# from sklearn.preprocessing import OneHotEncoder


# ohe_encoder = OneHotEncoder(
#     sparse_output=False,
#     handle_unknown="ignore"
# )

# # Fit on training data
# trained_encoder = ohe_encoder.fit_transform(X_train[multi_class_nominal_col])

# # Transform test data
# # Using fit_transform on the test set causes data leakage and can produce a different encoding.
# tested_encoder = ohe_encoder.transform(X_test[multi_class_nominal_col])

# # Column names
# ohe_cols = ohe_encoder.get_feature_names_out(multi_class_nominal_col)

# # Create DataFrames
# train_ohe = pd.DataFrame(
#     trained_encoder,
#     columns=ohe_cols,
#     index=X_train.index
# )

# test_ohe = pd.DataFrame(
#     tested_encoder,
#     columns=ohe_cols,
#     index=X_test.index
# )

### **Column Transformer**
- Useful for combining ordinal and one hot encoding

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

ct = ColumnTransformer(
    transformers=[
        ('one_hot_encoder', OneHotEncoder(), multi_class_nominal_col),
        ('ordinal_encoder', OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), binary_and_ordinal_col)],
    remainder="passthrough" # "Leave every other column unchanged."
)

X_train_processed = ct.fit_transform(X_train)
X_test_processed = ct.transform(X_test)

In [ ]:
# Turn the one-hot arrays into proper DataFrames with real column names
X_train_df = pd.DataFrame(
    X_train_processed,
    columns=ct.get_feature_names_out(),
    index=X_train.index
)

X_test_df = pd.DataFrame(
    X_test_processed,
    columns=ct.get_feature_names_out(),
    index=X_test.index
)

In [37]:
X_train_df.columns

Index(['one_hot_encoder__embark_town_Cherbourg',
       'one_hot_encoder__embark_town_Queenstown',
       'one_hot_encoder__embark_town_Southampton', 'ordinal_encoder__sex',
       'ordinal_encoder__adult_male', 'ordinal_encoder__alone',
       'remainder__pclass', 'remainder__age', 'remainder__sibsp',
       'remainder__parch', 'remainder__fare'],
      dtype='object')

In [ ]:
print(f"Shape before adding (train): {X_train_df.shape}")
print(f"Shape before adding (test): {X_test_df.shape}")

X_train_df[target] = y_train.values
X_test_df[target] = y_test.values

print(f"Shape after adding (train): {X_train_df.shape}")
print(f"Shape after adding (test): {X_test_df.shape}")

train_path = "X_train_encoded.csv"
test_path = "X_test_encoded.csv"
X_train_df.to_csv(train_path, index=False)
X_test_df.to_csv(test_path, index=False)
print(f"Saved files: {train_path}, {test_path}")

Shape before adding: (712, 12)
Shape before adding: (179, 12)
Shape after adding: (712, 12)
Shape after adding: (179, 12)


### **Using Pandas Mapping**

In [19]:
# level_encoder = {"Freshman" : 1, "Junior" : 2, "Sophomore" : 3, "Senior" : 4}
# student_df["Level"] = student_df["Level"].map(level_encoder)

### **Using Pandas Dummies**

In [20]:
student_df = pd.DataFrame({
    "Student": ["Alice", "Bob", "Charlie", "David", "Eva", "Frank"],
    "Age": [18, 19, 18, 20, 19, 21],
    "Grade": ["A", "B", "A", "C", "B", "A"],
    "Score": [92, 85, 90, 78, 88, 95]
})

print(student_df)
student_df = pd.get_dummies(student_df, columns = ["Grade"])

student_df.head()

   Student  Age Grade  Score
0    Alice   18     A     92
1      Bob   19     B     85
2  Charlie   18     A     90
3    David   20     C     78
4      Eva   19     B     88
5    Frank   21     A     95


,Student,Age,Score,Grade_A,Grade_B,Grade_C
0,Alice,18,92,True,False,False
1,Bob,19,85,False,True,False
2,Charlie,18,90,True,False,False
3,David,20,78,False,False,True
4,Eva,19,88,False,True,False
